#Part 4 — GenAI:
turn call-centre notes into structured reasons, then recommend

Task 4.1 — One validated call

In [12]:
# Install the Gemini API library and Pydantic
!pip install -q google-genai pydantic

In [13]:
import json
import pandas as pd
from google import genai
from google.genai import types
from pydantic import BaseModel, Field

In [ ]:
# Gemini Flash model used in Session 22.1
MODEL_NAME = "gemini-3.5-flash"

# Ask for the API key at runtime
# The key will not be written directly into the notebook code
API_KEY = input("Paste your Gemini API key: ")

# Create Gemini API client
client = genai.Client(api_key=API_KEY)

In [15]:
# Define the required JSON response structure
SCHEMA = {
    "type": "object",

    "properties": {

        # Main reason for missing the appointment
        "root_cause": {
            "type": "string"
        },

        # Confidence score between 0 and 1
        "confidence": {
            "type": "number"
        },

        # Short explanation (maximum 25 words)
        "summary": {
            "type": "string"
        },

        # Suggested action for the clinic
        "recommended_action": {
            "type": "string"
        }
    },

    # All four fields must be present
    "required": [
        "root_cause",
        "confidence",
        "summary",
        "recommended_action"
    ]
}

In [16]:
# Tell Gemini how it must behave
SYSTEM_TEXT = """
You are an expert healthcare appointment follow-up analyst.

Your job is to analyse call-centre notes
and identify why a patient missed an appointment.

Use only information supported by the note.
Do not invent missing details.
Return structured JSON according to the supplied schema.
Keep summaries within 25 words.
"""

In [17]:
# Define the allowed root cause categories
ALLOWED_CAUSES = [
    "forgot_or_no_reminder",
    "distance_or_transport",
    "cost_or_fee",
    "illness_or_emergency",
    "work_or_schedule",
    "other"
]

# Function to construct the CRAFT prompt
def build_prompt(note_text):

    prompt = f"""
    CONTEXT:
    CareLoop Clinics wants to understand why patients
    miss their scheduled appointments.

    ROLE:
    Act as an expert healthcare appointment analyst.

    ACTION:
    Read the following call-centre note and identify
    the most likely root cause.

    Allowed root_cause values:
    {ALLOWED_CAUSES}

    If the reason is unclear or ambiguous, choose "other".
    Do not guess or invent information.

    TONE:
    Professional, factual and concise.

    FORMAT:
    Return exactly four fields in JSON:
    root_cause, confidence, summary, recommended_action.

    The summary must contain at most 25 words.
    Confidence must be between 0 and 1.

    CALL-CENTRE NOTE:
    {note_text}
    """

    return prompt

In [18]:
# Function to analyse one call-centre note using Gemini
def analyse_note(note_text):

    # Build the CRAFT prompt for the supplied note
    prompt = build_prompt(note_text)

    # Send the prompt to Gemini
    response = client.models.generate_content(

        # Select the Gemini Flash model
        model=MODEL_NAME,

        # Supply the constructed prompt
        contents=prompt,

        # Configure Gemini's response
        config=types.GenerateContentConfig(

            # Apply the system instruction
            system_instruction=SYSTEM_TEXT,

            # Make responses as consistent as possible
            temperature=0,

            # Allow enough output tokens for a complete response
            max_output_tokens=1000,

            # Request JSON output
            response_mime_type="application/json",

            # Enforce the four-field JSON schema
            response_schema=SCHEMA,

            # Use minimal thinking as specified in Session 22.1
            thinking_config=types.ThinkingConfig(
                thinking_level="minimal"
            )
        )
    )

    # Convert Gemini's JSON text into a Python dictionary
    data = json.loads(response.text)

    # Return the structured result
    return data

In [27]:
# Read the follow-up notes CSV
notes_df = pd.read_csv("/content/followup_notes.csv")

print(notes_df.columns.tolist())
display(notes_df.head(10))



['note_id', 'appointment_id', 'note_text']


,note_id,appointment_id,note_text
0,N001,A0872,Patient said she never got any reminder messag...
1,N002,A0897,Lives far from the clinic and could not find a...
2,N003,A0526,Patient felt the consultation fee was too high...
3,N004,A0926,NaN
4,N005,A0534,"Had high fever and body ache on the day, staye..."
5,N006,A0507,Manager called an urgent meeting at the office...
6,N007,A0333,"Bhool gaya, SMS nahi aaya tha. Dobara booking ..."
7,N008,A0222,"City bus strike that day, no auto available ne..."
8,N009,A0977,"Phone was switched off, could not reach the pa..."
9,N010,A0596,Had a shift change at the factory and could no...


In [28]:
# Find note N001
note_text = notes_df.loc[
    notes_df["note_id"] == "N001",
    "note_text"
].iloc[0]

# Analyse N001 using Gemini
result = analyse_note(note_text)

# Display the structured dictionary
print(result)

{'root_cause': 'forgot_or_no_reminder', 'confidence': 0.95, 'summary': 'Patient forgot the appointment because she did not receive a reminder message and requested to rebook for next week.', 'recommended_action': "Reschedule the patient's appointment for next week and verify that automated SMS/email reminders are correctly configured for her account."}


Task 4.2 — Validate and process the 10 notes

In [21]:
from pydantic import BaseModel, Field, ValidationError

class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)

In [22]:
def analyse_note_with_raw(note_text):
    prompt = build_prompt(note_text)

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=1000,
            response_mime_type="application/json",
            response_schema=SCHEMA,
            thinking_config=types.ThinkingConfig(
                thinking_level="minimal"
            )
        )
    )

    raw_text = response.text
    data = json.loads(raw_text)

    return data, raw_text

In [29]:

import time
import pandas as pd

results = []
api_calls_made_4_2 = 0

for _, row in notes_df.head(10).iterrows():

    note_text = row["note_text"]

    # Skip blank notes without calling Gemini
    if pd.isna(note_text) or str(note_text).strip() == "":
        results.append({
            "note_id": row["note_id"],
            "appointment_id": row["appointment_id"],
            "root_cause": "other",
            "confidence": 0.0,
            "summary": "No note recorded",
            "recommended_action": "Re-call patient",
            "validation_status": "skipped_blank",
            "raw_response": ""
        })
        continue

    try:
        data, raw_text = analyse_note_with_raw(str(note_text))
        api_calls_made_4_2 += 1

        result = {
            "note_id": row["note_id"],
            "appointment_id": row["appointment_id"],
            "root_cause": data.get("root_cause", ""),
            "confidence": data.get("confidence", ""),
            "summary": data.get("summary", ""),
            "recommended_action": data.get("recommended_action", ""),
            "validation_status": "",
            "raw_response": raw_text
        }

        # Validate the response
        insight = FollowupInsight(**data)

        if insight.root_cause not in ALLOWED_CAUSES:
            raise ValueError("Root cause is not in the allowed categories")

        result["validation_status"] = "valid"

    except Exception as e:
        result["validation_status"] = "invalid"
        print(f"Validation/API error for {row['note_id']}: {e}")

        # Preserve the raw response whenever one was received.
        if "result" not in locals() or result.get("note_id") != row["note_id"]:
            result = {
                "note_id": row["note_id"],
                "appointment_id": row["appointment_id"],
                "root_cause": "",
                "confidence": "",
                "summary": "",
                "recommended_action": "",
                "validation_status": "invalid",
                "raw_response": locals().get("raw_text", "")
            }

    results.append(result)

    # Wait between nonblank-note API calls
    time.sleep(13)

notes_analysed = pd.DataFrame(results)

notes_analysed.to_csv("notes_analysed.csv", index=False)

print("API calls made in Task 4.2:", api_calls_made_4_2)
print("Output shape:", notes_analysed.shape)
display(notes_analysed)

API calls made in Task 4.2: 9
Output shape: (10, 8)


,note_id,appointment_id,root_cause,confidence,summary,recommended_action,validation_status,raw_response
0,N001,A0872,forgot_or_no_reminder,0.95,Patient missed the appointment because she for...,Reschedule the patient's appointment for next ...,valid,"{""root_cause"":""forgot_or_no_reminder"",""confide..."
1,N002,A0897,distance_or_transport,0.95,Patient missed the appointment due to living f...,Offer telehealth options or assist with schedu...,valid,"{""root_cause"":""distance_or_transport"",""confide..."
2,N003,A0526,cost_or_fee,0.95,Patient skipped the appointment due to high co...,"Provide information regarding payment plans, f...",valid,"{""root_cause"":""cost_or_fee"",""confidence"":0.95,..."
3,N004,A0926,other,0.00,No note recorded,Re-call patient,skipped_blank,
4,N005,A0534,illness_or_emergency,1.00,Patient missed the appointment due to a high f...,Follow up with the patient in a few days to ch...,valid,"{""root_cause"":""illness_or_emergency"",""confiden..."
5,N006,A0507,work_or_schedule,0.95,The patient missed the appointment due to an u...,Reschedule the patient's appointment to an eve...,valid,"{\n ""root_cause"": ""work_or_schedule"",\n ""con..."
6,N007,A0333,forgot_or_no_reminder,0.95,Patient forgot the appointment because they di...,Reschedule the appointment and verify the pati...,valid,"{\n ""root_cause"": ""forgot_or_no_reminder"",\n ..."
7,N008,A0222,distance_or_transport,0.95,The patient missed the appointment due to a ci...,Offer telehealth options or assist with altern...,valid,"{\n ""root_cause"": ""distance_or_transport"",\n ..."
8,N009,A0977,other,0.90,Unable to reach the patient after three calls ...,Attempt contact via alternative methods such a...,valid,"{""root_cause"":""other"",""confidence"":0.9,""summar..."
9,N010,A0596,work_or_schedule,0.95,The patient missed the appointment due to an u...,Reschedule the patient's appointment for an av...,valid,"{\n ""root_cause"": ""work_or_schedule"",\n ""con..."


In [41]:
print("Rows and columns:", notes_analysed.shape)
print("\nValidation status:")
print(notes_analysed["validation_status"].value_counts())

print("\nN004:")
display(notes_analysed[notes_analysed["note_id"] == "N004"])

print("\nRoot cause counts:")
print(notes_analysed["root_cause"].value_counts())

Rows and columns: (10, 8)

Validation status:
validation_status
valid            9
skipped_blank    1
Name: count, dtype: int64

N004:


,note_id,appointment_id,root_cause,confidence,summary,recommended_action,validation_status,raw_response
3,N004,A0926,other,0.0,No note recorded,Re-call patient,skipped_blank,



Root cause counts:
root_cause
forgot_or_no_reminder    2
distance_or_transport    2
other                    2
work_or_schedule         2
cost_or_fee              1
illness_or_emergency     1
Name: count, dtype: int64


Task 4.3 — Test validation and compare prompts

In [31]:
# Test 1: Confidence greater than 1
try:
    FollowupInsight(
        root_cause="other",
        confidence=1.7,
        summary="Patient had an unclear reason",
        recommended_action="Call patient again"
    )
except ValidationError as e:
    print("Test 1 passed: Invalid confidence rejected")
    print(e)


# Test 2: Missing recommended_action
try:
    FollowupInsight(
        root_cause="other",
        confidence=0.8,
        summary="Patient had an unclear reason"
    )
except ValidationError as e:
    print("\nTest 2 passed: Missing field rejected")
    print(e)

Test 1 passed: Invalid confidence rejected
1 validation error for FollowupInsight
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.7, input_type=float]
    For further information visit https://errors.pydantic.dev/2.14/v/less_than_equal

Test 2 passed: Missing field rejected
1 validation error for FollowupInsight
recommended_action
  Field required [type=missing, input_value={'root_cause': 'other', '... had an unclear reason'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing


In [32]:
def compare_prompt(note_text, version):

    if version == "V1":
        prompt = f"Classify this note:\n\n{note_text}"
    else:
        prompt = build_prompt(note_text)

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=1000,
            response_mime_type="application/json",
            response_schema=SCHEMA,
            thinking_config=types.ThinkingConfig(
                thinking_level="minimal"
            )
        )
    )

    return json.loads(response.text)

In [33]:
comparison_results = []

selected_notes = notes_df[
    notes_df["note_id"].isin(["N007", "N009"])
]

api_calls_made_4_3 = 0

for _, row in selected_notes.iterrows():

    note_text = str(row["note_text"])

    for version in ["V1", "V2"]:

        try:
            data = compare_prompt(note_text, version)
            api_calls_made_4_3 += 1

            root_cause = data.get("root_cause", "")

            comparison_results.append({
                "note_id": row["note_id"],
                "prompt_version": version,
                "root_cause": root_cause,
                "allowed_category": root_cause in ALLOWED_CAUSES
            })

        except Exception as e:
            print(row["note_id"], version, "Error:", e)

            comparison_results.append({
                "note_id": row["note_id"],
                "prompt_version": version,
                "root_cause": "ERROR",
                "allowed_category": False
            })

        time.sleep(13)

comparison_df = pd.DataFrame(comparison_results)

display(comparison_df)

,note_id,prompt_version,root_cause,allowed_category
0,N007,V1,Patient forgot the appointment and did not rec...,False
1,N007,V2,forgot_or_no_reminder,True
2,N009,V1,Unreachable / Phone Switched Off,False
3,N009,V2,other,True


In [40]:
api_calls_total = 1 + api_calls_made_4_2 + api_calls_made_4_3

print("Task 4.1 calls:", 1)
print("Task 4.2 calls:", api_calls_made_4_2)
print("Task 4.3 calls:", api_calls_made_4_3)
print("Total API calls:", api_calls_total)

Task 4.1 calls: 1
Task 4.2 calls: 9
Task 4.3 calls: 4
Total API calls: 14


Task 4.4 — Merge, analyse and prepare submission

In [36]:
appointments_df = pd.read_csv("appointments_clean.csv")

merged = notes_analysed.merge(
    appointments_df[["appointment_id", "reminder_sent"]],
    on="appointment_id",
    how="left"
)

print("Merged shape:", merged.shape)

display(
    merged[
        ["note_id", "appointment_id", "root_cause", "reminder_sent"]
    ]
)

Merged shape: (10, 9)


,note_id,appointment_id,root_cause,reminder_sent
0,N001,A0872,forgot_or_no_reminder,No
1,N002,A0897,distance_or_transport,No
2,N003,A0526,cost_or_fee,Yes
3,N004,A0926,other,No
4,N005,A0534,illness_or_emergency,Yes
5,N006,A0507,work_or_schedule,Yes
6,N007,A0333,forgot_or_no_reminder,No
7,N008,A0222,distance_or_transport,Yes
8,N009,A0977,other,Yes
9,N010,A0596,work_or_schedule,No


In [42]:
cross_tab = pd.crosstab(
    merged["root_cause"],
    merged["reminder_sent"]
)

display(cross_tab)

reminder_sent,No,Yes
root_cause,,
cost_or_fee,0,1
distance_or_transport,1,1
forgot_or_no_reminder,2,0
illness_or_emergency,0,1
other,1,1
work_or_schedule,1,1


In [43]:
print("Reminder status counts:")
print(merged["reminder_sent"].value_counts(dropna=False))

Reminder status counts:
reminder_sent
No     5
Yes    5
Name: count, dtype: int64


In [45]:
notes_analysed.to_csv("notes_analysed.csv", index=False)
merged.to_csv("notes_appointments_merged.csv", index=False)
cross_tab.to_csv("root_cause_reminder_crosstab.csv")

print("All Task 4.4 files saved.")

All Task 4.4 files saved.
